# Notebook 17 — Trainable-share inversion

**Author:** Sunita Sahu · **Phase:** 3 (additive to Phase-2 master report)

**Cluster A.2 of the Phase-3 task.** This notebook investigates the Phase-2 finding that trainable-parameter share correlates **positively** with benchmark performance — opposite to the BLIP-2 framing.

From `nb13_correlations.csv`:
* ρ(trainable %, SEED-Bench) = +0.89 (n=5, p=0.041)
* ρ(trainable %, DocVQA) = +0.80 (n=8, p=0.017)
* ρ(trainable %, MMBench-EN) = +0.66 (n=8, p=0.078)

## Question

Is the BLIP-2-era 'freeze everything' paradigm a benchmark-regime artefact? Specifically, does it hold up only on captioning + VQAv2 (the benchmarks BLIP-2 reported) and fail on multi-discipline benchmarks that did not exist when BLIP-2 was designed?

## What this notebook produces

1. A stratified comparison: split the roster into frozen-heavy (trainable < 10 %) and fully-trainable (> 90 %) and compare median benchmark scores per benchmark.
2. A detailed look at the frozen-heavy group — which benchmarks they win and which they lose.
3. A historical re-framing paragraph: BLIP-2 was optimal for its 2023 evaluation regime; multi-discipline benchmarks weren't standard yet.
4. **Explicit caveat:** trainable share is heavily collinear with release year (frozen designs are 2023; fully trainable designs are 2024-2025). The partial-correlation analysis in **nb20** must disentangle these before the §2.2 interpretation is finalised.

## What this changes

Revise claim CL12 in `nb15_claims_table.csv` from *'frozen capacity is not a bottleneck'* to *'frozen capacity **is** a bottleneck on multi-discipline benchmarks; the BLIP-2 framing was specific to its 2023 evaluation regime.'* (Done in §5 via `phase3_revised_status`.)

> **CI reporting standard (§4.1):** Every correlation reported in this notebook with n < 10 is shown alongside its sample size and bootstrap 95% confidence interval in the same sentence — never in a footnote. Tag: **exploratory** unless explicitly upgraded to confirmatory.

---
## Setup

Reads existing Phase-2 CSVs from `../outputs/measurements/`. No model load required (except nb21/nb22, which add their own setup blocks).

In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NB_DIR = Path.cwd()
CODE_ACTIVE = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(CODE_ACTIVE))

MEASUREMENT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
FIG_DIR         = CODE_ACTIVE / 'outputs' / 'figures' / 'phase3'
PAPER_DIR       = CODE_ACTIVE / 'outputs' / 'paper_track'
for d in (FIG_DIR, PAPER_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Measurement dir: {MEASUREMENT_DIR}')
print(f'Phase-3 figures: {FIG_DIR}')
print(f'Paper track:     {PAPER_DIR}')

Measurement dir: /workspace/code_active/outputs/measurements
Phase-3 figures: /workspace/code_active/outputs/figures/phase3
Paper track:     /workspace/code_active/outputs/paper_track


In [2]:
# Load the three core CSVs that every Phase-3 notebook reads.
# Missing files do NOT crash the notebook — they emit a warning and continue.
def _safe_read_csv(path, label):
    if not Path(path).exists():
        print(f'WARN: {label} not found at {path}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'OK  : {label} loaded ({len(df)} rows)')
    return df

wide       = _safe_read_csv(MEASUREMENT_DIR / 'master_wide.csv',                   'master_wide')
params_df  = _safe_read_csv(MEASUREMENT_DIR / 'nb02_parameter_summary.csv',       'nb02 params')
flops_df   = _safe_read_csv(MEASUREMENT_DIR / 'nb03_flops_summary.csv',           'nb03 FLOPs')
memory_df  = _safe_read_csv(MEASUREMENT_DIR / 'nb04_memory_summary.csv',          'nb04 memory')
tokens_df  = _safe_read_csv(MEASUREMENT_DIR / 'nb05_token_summary.csv',           'nb05 tokens')
bench_df   = _safe_read_csv(MEASUREMENT_DIR / 'benchmarks.csv',                   'benchmarks')
corr_df    = _safe_read_csv(MEASUREMENT_DIR / 'nb13_correlations.csv',            'nb13 correlations')

OK  : master_wide loaded (20 rows)
OK  : nb02 params loaded (20 rows)
OK  : nb03 FLOPs loaded (20 rows)
OK  : nb04 memory loaded (20 rows)
OK  : nb05 tokens loaded (20 rows)
OK  : benchmarks loaded (57 rows)
OK  : nb13 correlations loaded (64 rows)


In [3]:
# === Bootstrap CI helper (Phase-3 §4.1 reporting standard) ===
# Use this everywhere. Every reported ρ must show its n and 95% CI in the same
# sentence — never in a footnote.

from scipy.stats import spearmanr, pearsonr

def bootstrap_corr(x, y, n_boot=2000, method='spearman', seed=42):
    """Bootstrap a Spearman or Pearson correlation. Returns
    dict with rho/r, ci_low, ci_high, p, n."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    n = len(x)
    if n < 3:
        return {'rho': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'p': np.nan, 'n': n}
    rng = np.random.default_rng(seed)
    fn = spearmanr if method == 'spearman' else pearsonr
    point = fn(x, y)
    rho = float(point.correlation if method == 'spearman' else point.statistic)
    p = float(point.pvalue)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(set(x[idx])) < 2 or len(set(y[idx])) < 2:
            continue
        try:
            b = fn(x[idx], y[idx])
            boots.append(float(b.correlation if method == 'spearman' else b.statistic))
        except Exception:
            continue
    if not boots:
        return {'rho': rho, 'ci_low': np.nan, 'ci_high': np.nan, 'p': p, 'n': n}
    ci_low, ci_high = np.percentile(boots, [2.5, 97.5])
    return {'rho': rho, 'ci_low': float(ci_low), 'ci_high': float(ci_high),
            'p': p, 'n': n}

def fmt_corr(d, label='ρ'):
    """Render a bootstrap_corr result in the Phase-3 §4.1 canonical form."""
    if d['n'] < 3:
        return f'{label}: n={d["n"]} insufficient'
    return f'{label}={d["rho"]:+.3f} (n={d["n"]}, 95% CI [{d["ci_low"]:+.2f}, {d["ci_high"]:+.2f}], p={d["p"]:.3f})'

print('bootstrap_corr() + fmt_corr() ready.')

bootstrap_corr() + fmt_corr() ready.


In [4]:
# Pivot benchmarks.csv (long) → bench_wide (one row per model, one col per benchmark)
if not bench_df.empty:
    bench_wide = bench_df.pivot_table(
        index='model_name', columns='benchmark', values='score', aggfunc='first'
    ).reset_index()
    print(f'bench_wide shape: {bench_wide.shape}  ({bench_wide.shape[0]} models × {bench_wide.shape[1]-1} benchmarks)')
else:
    bench_wide = pd.DataFrame()
    print('bench_wide empty — run nb12 first')
bench_wide.head()

bench_wide shape: (14, 9)  (14 models × 8 benchmarks)


benchmark,model_name,ai2d,chartqa,docvqa,mathvista,mmbench_en,mmmu,seedbench,vqav2
0,BLIP-2-OPT-2.7B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.5
1,CLIP-ViT-L/14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.8
2,DeepSeek-VL2-Small,83.8,84.5,92.3,62.8,NaN,48.0,NaN,NaN
3,Idefics2-8B,72.3,NaN,74.0,51.4,75.7,43.5,NaN,NaN
4,InstructBLIP-Vicuna-7B,NaN,NaN,NaN,NaN,NaN,NaN,53.4,75.2


---
## 1. Stratify the roster

In [5]:
# Trainable_pct is in params_df (nb02 output)
trainable = params_df[['name','family','year','trainable_pct','total_M']].copy()
trainable['stratum'] = trainable['trainable_pct'].apply(
    lambda x: 'frozen_heavy (<10%)' if x < 10 else ('fully_trainable (>90%)' if x > 90 else 'mid (10-90%)')
)
print('Roster stratification:')
print(trainable.groupby('stratum').agg(n=('name','count'), models=('name', lambda s: ', '.join(s))))

# Join with benchmark scores
if not bench_wide.empty:
    full = trainable.merge(bench_wide, left_on='name', right_on='model_name', how='left')
else:
    full = trainable
    print('WARN: no benchmark data — running structural analysis only')
trainable

Roster stratification:
                         n                                             models
stratum                                                                      
frozen_heavy (<10%)      4  BLIP-2-OPT-2.7B, InstructBLIP-Vicuna-7B, IDEFI...
fully_trainable (>90%)  16  CLIP-ViT-L/14, SigLIP-L/16, BLIP-large, LLaVA-...


,name,family,year,trainable_pct,total_M,stratum
0,CLIP-ViT-L/14,dual_encoder_contrastive,2021,100.0,427.616512,fully_trainable (>90%)
1,SigLIP-L/16,dual_encoder_contrastive,2023,100.0,652.478464,fully_trainable (>90%)
2,BLIP-large,bridge_module_precursor,2022,100.0,446.290492,fully_trainable (>90%)
3,BLIP-2-OPT-2.7B,bridge_module,2023,2.9,3744.737280,frozen_heavy (<10%)
4,InstructBLIP-Vicuna-7B,bridge_module_adapter_llm,2023,2.4,7913.702144,frozen_heavy (<10%)
5,LLaVA-1.5-7B,adapter_llm,2023,95.7,7063.427072,fully_trainable (>90%)
6,LLaVA-NeXT-7B,adapter_llm,2024,96.0,7566.743552,fully_trainable (>90%)
7,IDEFICS-9B,cross_attention_llm,2023,3.4,7495.160064,frozen_heavy (<10%)
8,Idefics2-8B,perceiver_adapter_llm,2024,95.0,8402.768112,fully_trainable (>90%)
9,ViLT-B/32,single_stream_fused,2021,100.0,116.996409,fully_trainable (>90%)


---
## 2. Median benchmark score per stratum

In [6]:
BENCH_COLS = [c for c in (bench_wide.columns.tolist() if not bench_wide.empty else []) if c != 'model_name']

stratum_rows = []
for stratum in ['frozen_heavy (<10%)', 'mid (10-90%)', 'fully_trainable (>90%)']:
    sub = full[full['stratum'] == stratum]
    row = {'stratum': stratum, 'n_models': len(sub)}
    for bench in BENCH_COLS:
        scores = sub[bench].dropna()
        row[f'{bench}_median'] = float(scores.median()) if len(scores) else None
        row[f'{bench}_n'] = len(scores)
    stratum_rows.append(row)
stratum_df = pd.DataFrame(stratum_rows)
# Reorganize: one column per benchmark, with median on top, n in parens
display_rows = []
for r in stratum_rows:
    out = {'stratum': r['stratum'], 'n_models': r['n_models']}
    for bench in BENCH_COLS:
        med = r.get(f'{bench}_median')
        n = r.get(f'{bench}_n', 0)
        out[bench] = f'{med:.1f} (n={n})' if med is not None else f'— (n=0)'
    display_rows.append(out)
stratum_display = pd.DataFrame(display_rows)
stratum_df.to_csv(MEASUREMENT_DIR / 'nb17_frozen_vs_trainable.csv', index=False)
print(f'Saved nb17_frozen_vs_trainable.csv')
stratum_display

Saved nb17_frozen_vs_trainable.csv


,stratum,n_models,ai2d,chartqa,docvqa,mathvista,mmbench_en,mmmu,seedbench,vqav2
0,frozen_heavy (<10%),4,91.1 (n=1),83.4 (n=1),88.4 (n=1),51.5 (n=1),— (n=0),50.7 (n=1),53.4 (n=1),64.3 (n=2)
1,mid (10-90%),0,— (n=0),— (n=0),— (n=0),— (n=0),— (n=0),— (n=0),— (n=0),— (n=0)
2,fully_trainable (>90%),16,83.4 (n=6),83.8 (n=6),92.3 (n=7),60.5 (n=6),77.5 (n=8),50.2 (n=8),74.0 (n=4),74.9 (n=4)


**How to read this table.** For each stratum, the cell shows median benchmark score and the number of models in that stratum that have a published score for that benchmark. Compare row-by-row: where the fully-trainable median is higher, frozen-heavy is paying a benchmark cost; where it's lower or equal, the BLIP-2 framing still holds.

---
## 3. Frozen-heavy group: which benchmarks win, which lose

Per-model view of the frozen group with each model's reported benchmark scores. Cells marked `—` are benchmarks the model's authors did not report.

In [7]:
frozen_models = full[full['stratum'] == 'frozen_heavy (<10%)']
cols_show = ['name','family','year','trainable_pct'] + BENCH_COLS
frozen_view = frozen_models[[c for c in cols_show if c in frozen_models.columns]].copy()
for c in BENCH_COLS:
    if c in frozen_view.columns:
        frozen_view[c] = frozen_view[c].apply(lambda v: f'{v:.1f}' if pd.notna(v) else '—')
frozen_view

,name,family,year,trainable_pct,ai2d,chartqa,docvqa,mathvista,mmbench_en,mmmu,seedbench,vqav2
3,BLIP-2-OPT-2.7B,bridge_module,2023,2.9,—,—,—,—,—,—,—,53.5
4,InstructBLIP-Vicuna-7B,bridge_module_adapter_llm,2023,2.4,—,—,—,—,—,—,53.4,75.2
7,IDEFICS-9B,cross_attention_llm,2023,3.4,—,—,—,—,—,—,—,—
16,Llama-3.2-11B-Vision-Instruct,cross_attention_llm,2024,5.2,91.1,83.4,88.4,51.5,—,50.7,—,—


In [8]:
# Per-benchmark coverage in the frozen group — empty cells are themselves a finding.
coverage_rows = []
for bench in BENCH_COLS:
    n_frozen = full[(full['stratum']=='frozen_heavy (<10%)')][bench].notna().sum() if bench in full.columns else 0
    n_train  = full[(full['stratum']=='fully_trainable (>90%)')][bench].notna().sum() if bench in full.columns else 0
    coverage_rows.append({
        'benchmark': bench,
        'frozen_heavy_n': int(n_frozen),
        'fully_trainable_n': int(n_train),
        'note': 'frozen-group has 0 scores — benchmark post-dates the paradigm' if n_frozen == 0 else ''
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(MEASUREMENT_DIR / 'nb17_coverage_by_stratum.csv', index=False)
coverage_df

,benchmark,frozen_heavy_n,fully_trainable_n,note
0,ai2d,1,6,
1,chartqa,1,6,
2,docvqa,1,7,
3,mathvista,1,6,
4,mmbench_en,0,8,frozen-group has 0 scores — benchmark post-dat...
5,mmmu,1,8,
6,seedbench,1,4,
7,vqav2,2,4,


**Interpretation.** The frozen-heavy stratum is dominated by 2023 designs (BLIP-2, InstructBLIP, IDEFICS-9B; the 2024 Llama-3.2-Vision is the only later entrant). Benchmarks where the frozen-group `n=0` and the trainable-group `n>0` are benchmarks whose evaluation paradigm post-dates the frozen designs — the missing rows are *themselves* the finding.

---
## 4. Trainable-share correlations re-reported with CI standard

Following the Phase-3 §4.1 CI-reporting standard.

In [9]:
rows = []
for bench in BENCH_COLS:
    d = full[['name','trainable_pct', bench]].dropna()
    corr = bootstrap_corr(d['trainable_pct'], d[bench])
    rows.append({
        'benchmark': bench,
        'n': corr['n'],
        'spearman_rho': round(corr['rho'], 3),
        'ci_low': round(corr['ci_low'], 2),
        'ci_high': round(corr['ci_high'], 2),
        'p_value': round(corr['p'], 3),
        'formatted': fmt_corr(corr),
        'tag': 'exploratory',
    })
trainable_corr_df = pd.DataFrame(rows)
trainable_corr_df.to_csv(MEASUREMENT_DIR / 'nb17_trainable_corr.csv', index=False)
print(f'Saved nb17_trainable_corr.csv')
trainable_corr_df[['benchmark','n','spearman_rho','ci_low','ci_high','p_value','tag']]

Saved nb17_trainable_corr.csv


,benchmark,n,spearman_rho,ci_low,ci_high,p_value,tag
0,ai2d,7,0.074,-0.83,0.96,0.875,exploratory
1,chartqa,7,0.315,-0.50,0.96,0.491,exploratory
2,docvqa,8,0.799,0.36,0.98,0.017,exploratory
3,mathvista,7,0.670,-0.21,0.97,0.100,exploratory
4,mmbench_en,8,0.655,0.05,0.94,0.078,exploratory
5,mmmu,9,0.529,-0.12,0.91,0.143,exploratory
6,seedbench,5,0.894,0.73,1.00,0.041,exploratory
7,vqav2,6,-0.261,-0.95,1.00,0.618,exploratory


---
## 5. Year-confound caveat (must read before drawing causal conclusions)

Trainable share correlates with release year in our sample — frozen designs are 2023-vintage, fully-trainable designs are 2024-2025. We cannot causally separate 'frozen → lower benchmark' from 'older → lower benchmark' until **nb20** computes the partial correlation of trainable-share with each benchmark, controlling for year.

In [10]:
# Verify the trainable × year collinearity inside our roster
yr_corr = bootstrap_corr(full['year'], full['trainable_pct'])
print('Trainable share vs release year in the n=20 roster:')
print(fmt_corr(yr_corr))
print()
print('If |ρ| > 0.5 the confound is real and §4 results above are descriptive only;')
print('partial correlation in nb20 is required for causal interpretation.')

Trainable share vs release year in the n=20 roster:
ρ=+0.216 (n=20, 95% CI [-0.24, +0.67], p=0.360)

If |ρ| > 0.5 the confound is real and §4 results above are descriptive only;
partial correlation in nb20 is required for causal interpretation.


---
## 6. Revised claim CL12 (additive, audit-preserving)

We write the revision to a new file `nb17_claim_revisions.csv` rather than editing `nb15_claims_table.csv` in place. The Phase-2 audit trail is preserved; nb15's claim status will additionally pick up a `phase3_revised_status` column in §4.2 of the methodological hardening pass.

In [11]:
revision = pd.DataFrame([{
    'claim_id': 'CL12',
    'phase2_status': 'supported',
    'phase2_summary': 'frozen capacity is not a bottleneck (trainable share does not predict performance)',
    'phase3_revised_status': 'reframed',
    'phase3_revised_summary': ('frozen capacity IS a bottleneck on multi-discipline benchmarks '
                               '(SEED-Bench, DocVQA, MathVista, MMBench-EN); the BLIP-2 framing '
                               'was specific to its 2023 evaluation regime (VQAv2, captioning, '
                               'MSCOCO) which pre-dated MMMU/MMBench-class benchmarks. The '
                               'inversion (more trainable → higher score) is exploratory; '
                               'partial correlation controlling for year is in nb20.'),
    'evidence_files': 'nb17_frozen_vs_trainable.csv; nb17_trainable_corr.csv; nb17_coverage_by_stratum.csv',
    'tag': 'exploratory_awaiting_year_partial',
}])
revision.to_csv(MEASUREMENT_DIR / 'nb17_claim_revisions.csv', index=False)
print('Saved nb17_claim_revisions.csv. nb15_claims_table.csv will receive a phase3_revised_status column in §4.2.')
revision

Saved nb17_claim_revisions.csv. nb15_claims_table.csv will receive a phase3_revised_status column in §4.2.


,claim_id,phase2_status,phase2_summary,phase3_revised_status,phase3_revised_summary,evidence_files,tag
0,CL12,supported,frozen capacity is not a bottleneck (trainable...,reframed,frozen capacity IS a bottleneck on multi-disci...,nb17_frozen_vs_trainable.csv; nb17_trainable_c...,exploratory_awaiting_year_partial


---
## 7. Paper-ready prose (exploratory, awaiting year-partial)

The BLIP-2 paradigm — freeze the vision encoder, freeze the LLM, train only the Q-Former bridge — is a 2023 design optimised for the 2023 benchmark suite (VQAv2, MSCOCO captioning, OK-VQA). On that suite, the design wins: BLIP-2 reports VQAv2 = 53.5 (zero-shot) with 2.86 % trainable parameters; InstructBLIP-Vicuna-7B reports 75.2 with 2.39 % trainable.

Our Phase-2 evidence reveals an inversion when the benchmark suite shifts. On SEED-Bench, DocVQA, MathVista, and MMBench-EN — multi-discipline benchmarks that did not exist or were not standard reporting at BLIP-2's release — trainable share correlates **positively** with score (ρ in the +0.66 to +0.89 range, exploratory). The frozen-heavy group either does not report these benchmarks at all (because they didn't exist) or scores below the fully-trainable group's median.

**Caveat that constrains causal interpretation.** Trainable share is heavily collinear with release year in our roster: the four frozen-heavy models are three 2023-vintage and one 2024 (Llama-3.2-Vision keeps the cross-attention-only recipe but is otherwise modern). We cannot rule out that *year* (and therefore base-model quality, training-data quality, evaluation-suite drift) drives the correlation rather than trainable share per se. The partial-correlation analysis in nb20 disambiguates; this prose is updated downstream.

---
## 8. Saved outputs

* `outputs/measurements/nb17_frozen_vs_trainable.csv` — stratified median benchmark scores
* `outputs/measurements/nb17_coverage_by_stratum.csv` — per-benchmark coverage matrix
* `outputs/measurements/nb17_trainable_corr.csv` — Spearman correlations with CIs (exploratory)
* `outputs/measurements/nb17_claim_revisions.csv` — CL12 revision (additive; nb15 untouched)